# Encoding Categorical Data: One-Hot Encoding & ColumnTransformer

This notebook demonstrates **One-Hot Encoding (OHE)** for nominal categorical data and introduces **ColumnTransformer** for streamlining multi-column transformation pipelines.
We will use a toy dataset `covid_toy.csv` which has missing values and categorical data.

**💡 Cell Explanation:** Import the foundational data manipulation libraries. `numpy` is for numerical operations, and `pandas` is for handling structured data (like DataFrames).

In [16]:
import numpy as np
import pandas as pd

### Core Concept: How One-Hot Encoding Works
- **Categorical Data Types**:
  - **Ordinal**: Has an intrinsic order (e.g., Excellent > Good). For ordinal data, we use **Ordinal Encoding**.
  - **Nominal**: No intrinsic order (e.g., Male and Female). If we assigned 0 and 1, the ML algorithm might incorrectly interpret one as quantitatively greater than the other.
- **The One-Hot Encoding solution**: create **one new column per category**. 
  - For example, if we have a Color column with Yellow, Blue, Red. Yellow = `[1,0,0]`, Blue = `[0,1,0]`, Red = `[0,0,1]`. Each row's string category is effectively converted into a **vector**.

**💡 Cell Explanation:** Import the specific scikit-learn tools needed for the manual approach. `SimpleImputer` fills in missing values, `OneHotEncoder` handles nominal data, and `OrdinalEncoder` handles ordinal data.

In [17]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

**💡 Cell Explanation:** Load the dataset `covid_toy.csv` into a Pandas DataFrame called `df`. This allows us to view and manipulate the data.

In [27]:
df = pd.read_csv('covid_toy.csv')

**💡 Cell Explanation:** Inspect the data! `df.head()` shows the first 5 rows, giving us a quick look at the columns (`age`, `gender`, `fever`, `cough`, `city`, `has_covid`). `df.shape` tells us there are 100 rows and 6 columns.

In [39]:
# df.head()
df.dtypes
# df.shape

age            int64
gender           str
fever        float64
cough            str
city             str
has_covid        str
dtype: object

**💡 Cell Explanation:** Check for missing values in each column. We can see that the `fever` column has 10 missing values (`NaN`s), which we'll need to fix using imputation.

In [30]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

**💡 Cell Explanation:** Separate the dataset into features (our inputs) and labels (our output: `has_covid`). Then, split the data into a training set (80%) to train the model, and a testing set (20%) to evaluate it. *Always split your data before applying transformations to prevent data leakage!*

In [42]:
from sklearn.model_selection import train_test_split
target=df['has_covid']
feature=df.drop(columns=['has_covid'])
X_train,X_test,y_train,y_test = train_test_split(feature,target,
                                                test_size=0.2)

## 1. Aam Zindagi (The Ordinary Way)

### Why not Pandas' `get_dummies`?
Although Pandas' `pd.get_dummies()` is convenient for quick data analysis, it is **not suitable for ML projects**.
**Reason**: Pandas does not 'remember' which column was placed in which position/order during encoding. Running it on test data could produce a different result since it doesn't retain a fixed mapping. `scikit-learn`'s `OneHotEncoder` remembers the fitted encoding mapping, making it suitable for proper ML project workflows.

### The Challenge of Manual scikit-learn Transformers
When applying `OneHotEncoder` to only specific columns, you must:
1. Extract just those columns.
2. Apply One-Hot Encoding to them.
3. **Rejoin** the encoded result back with the untouched columns to reconstruct the complete input dataset.

This manual process is tedious ('twisted work'). Let's see how it looks below.

**💡 Cell Explanation:** Let's take a look at our training features before any transformations. Notice the missing values in the `fever` column and the text data in `gender`, `cough`, and `city`.

In [32]:
X_train

23    Yes
75    Yes
86    Yes
44     No
28     No
     ... 
91    Yes
27     No
92     No
13    Yes
96    Yes
Name: has_covid, Length: 80, dtype: str

**💡 Cell Explanation:** **Step 1:** Fill the missing values in the `fever` column. We use `SimpleImputer`, which by default replaces `NaN` with the mean (average) value of that column. Notice how we `.fit_transform` on the train data, but only `.transform` on the test data!

In [43]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever

array([[100.70422535],
       [ 98.        ],
       [104.        ],
       [104.        ],
       [103.        ],
       [101.        ],
       [ 99.        ],
       [101.        ],
       [100.        ],
       [100.70422535],
       [104.        ],
       [ 98.        ],
       [ 99.        ],
       [ 99.        ],
       [100.70422535],
       [ 98.        ],
       [101.        ],
       [100.        ],
       [102.        ],
       [100.        ],
       [100.        ],
       [102.        ],
       [100.70422535],
       [101.        ],
       [100.        ],
       [100.        ],
       [102.        ],
       [101.        ],
       [100.70422535],
       [100.70422535],
       [103.        ],
       [ 98.        ],
       [101.        ],
       [101.        ],
       [ 98.        ],
       [ 98.        ],
       [ 98.        ],
       [101.        ],
       [101.        ],
       [102.        ],
       [104.        ],
       [103.        ],
       [102.        ],
       [103

**💡 Cell Explanation:** **Step 2:** Convert the ordinal categorical column `cough` into numbers. Since 'Strong' is worse than 'Mild', we pass the specific order to `OrdinalEncoder` so it assigns values logically (e.g., Mild=0, Strong=1).

In [45]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough

array([[0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [1.],
       [1.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [0.],
       [1.],
       [0.],
       [0.],
       [0.],
       [0.],
       [1.],

### The Dummy Variable Trap
- **The Problem**: if you have N categories (and thus N columns), you should **drop one of those columns** leaving **N − 1 columns** to represent all N categories.
- **Why? (Multicollinearity)**: If you add up the values of the N columns for any row, the sum **always equals 1**. This is a direct mathematical relationship between the input columns, which shouldn't exist because input columns should be independent. This causes issues for **Linear Models**.
- **The fix**: drop one column entirely (using `drop='first'`). The dropped category can be inferred when all remaining columns are 0 — no information is lost.

### Avoiding the Sparse Matrix
- `OneHotEncoder` produces a **sparse matrix** by default. By passing `sparse=False` (or `sparse_output=False`), it returns a plain NumPy array rather than a sparse matrix, eliminating the need for the separate `.toarray()` conversion step.

**💡 Cell Explanation:** **Step 3:** Convert the nominal categorical columns `gender` and `city` into numbers. We use One-Hot Encoding to create separate binary columns for each category. We also use `drop='first'` to avoid the Dummy Variable Trap and `sparse=False` to get a normal array.

In [58]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city

array([[1., 0., 1., 0.],
       [0., 0., 0., 1.],
       [1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 0.],
       [1., 1., 0., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 0., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 0., 0.],
       [1., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [1., 1., 0., 0.],
       [0., 0., 0., 1.],
       [1., 0., 1., 0.],


**💡 Cell Explanation:** **Step 4:** Extract the remaining numerical column (`age`) that doesn't require any preprocessing. We need to isolate it before we stitch everything back together.

In [62]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values


# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age

array([[82],
       [ 5],
       [25],
       [54],
       [16],
       [42],
       [72],
       [47],
       [11],
       [20],
       [12],
       [73],
       [66],
       [60],
       [84],
       [80],
       [81],
       [13],
       [69],
       [19],
       [80],
       [64],
       [34],
       [19],
       [19],
       [11],
       [24],
       [38],
       [75],
       [42],
       [83],
       [31],
       [51],
       [68],
       [12],
       [26],
       [40],
       [15],
       [65],
       [ 5],
       [75],
       [48],
       [82],
       [46],
       [65],
       [25],
       [19],
       [83],
       [23],
       [ 8],
       [50],
       [20],
       [69],
       [27],
       [74],
       [ 5],
       [60],
       [10],
       [38],
       [16],
       [14],
       [56],
       [59],
       [49],
       [24],
       [27],
       [69],
       [10],
       [20],
       [34],
       [64],
       [14],
       [51],
       [ 6],
       [65],
       [83],
       [81],

**💡 Cell Explanation:** **Step 5:** Stitch it all back together! We use `np.concatenate` to horizontally stack (`axis=1`) the age array, imputed fever array, encoded gender/city array, and encoded cough array into one final, machine-learning-ready matrix.

In [63]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

---
## 2. Mentos Zindagi (The Smart Way using ColumnTransformer)

A cleaner solution — **`ColumnTransformer`** — allows different transformers to be applied to different columns in a single line of code as part of a unified pipeline.

**Why is it better?**
- **Clean & Concise:** No need to manually extract, transform, and horizontally stack (`np.c_` or `np.concatenate`) columns.
- **Less Error-Prone:** The same transformations are guaranteed to be applied to the test set seamlessly.
- **Pipeline Friendly:** It integrates perfectly with scikit-learn's `Pipeline`.

**💡 Cell Explanation:** Import the `ColumnTransformer` class. This is the star of the 'Mentos Zindagi' approach, allowing us to define all those tedious transformations in one single step.

In [66]:
from sklearn.compose import ColumnTransformer

**💡 Cell Explanation:** Define our pipeline! We pass a list of tuples to `transformers`. Each tuple specifies: a name, the transformer object, and the list of columns to apply it to. `remainder='passthrough'` tells it to keep any unspecified columns (like `age`) rather than dropping them.

In [67]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

**💡 Cell Explanation:** Execute the entire pipeline on the training data in one go! `fit_transform` learns the parameters (like the mean for fever, and categories for city/gender/cough) and transforms the data simultaneously. The shape `(80, 7)` shows we successfully processed the data.

In [70]:
X_trained_trans=transformer.fit_transform(X_train)
X_train_transformed


array([[ 82.        , 100.70422535,   1.        ,   0.        ,
          1.        ,   0.        ,   0.        ],
       [  5.        ,  98.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   1.        ],
       [ 25.        , 104.        ,   1.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 54.        , 104.        ,   0.        ,   0.        ,
          1.        ,   0.        ,   1.        ],
       [ 16.        , 103.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 42.        , 101.        ,   1.        ,   1.        ,
          0.        ,   0.        ,   0.        ],
       [ 72.        ,  99.        ,   1.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [ 47.        , 101.        ,   1.        ,   0.        ,
          0.        ,   0.        ,   1.        ],
       [ 11.        , 100.        ,   1.        ,   0.        ,
          0.    

**💡 Cell Explanation:** Seamlessly apply the exact same learned transformations to the test data using `.transform()`. This prevents any data leakage and guarantees the test set is shaped identically to the training set.

In [72]:
X_test_trans=transformer.transform(X_test)
X_test_trans.shape

(20, 7)

### Handling Columns with Many Categories (High Cardinality)
If a categorical column (like Car Brand) has many distinct categories (e.g., 32 brands), applying OHE directly would create 32 new columns, significantly increasing the dataset's **dimensionality** and slowing down processing.

**The Solution:** Keep only the **most frequent categories** as individual columns, and merge all remaining (rare) categories into a single new category called **'Others.'**

**Step-by-step logic:**
1. **Get counts**: `counts = df['Brand'].value_counts()`
2. **Define a threshold**: e.g., categories with fewer than 100 occurrences are rare.
3. **Identify rare categories**: `rare_brands = counts[counts < 100].index`
4. **Replace rare categories with 'Other'**: `df['Brand'] = df['Brand'].replace(rare_brands, 'Other')`
5. **Apply One-Hot Encoding** on the now-simplified column.